In [ ]:
from pathlib import Path
import csv, hashlib, json

roots = list(Path('/kaggle/input').rglob('FULL_RUN_STATUS.json'))
assert len(roots) == 1, roots
artifact_root = roots[0].parent
status = json.loads(roots[0].read_text())
assert status['status'] == 'COMPLETE'
assert status['completed_folds'] == 5
assert status['completed_p0_cases'] == 40
assert status['completed_retrospective_cases'] == 40
assert status['failed_cases'] == 0

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

selected = []
for path in artifact_root.rglob('*'):
    if not path.is_file():
        continue
    relative = path.relative_to(artifact_root).as_posix()
    kind = None
    if path.name == 'P0_float32.npy': kind = 'HELD_OUT_P0'
    elif path.name == 'best_training_loss.pt': kind = 'FOLD_CHECKPOINT'
    elif path.name == 'ALL_CASE_METHOD_METRICS.csv': kind = 'MAIN_METRICS'
    elif path.name == 'ALL_PCC_ROUND_TRAJECTORIES.csv': kind = 'MAIN_TRAJECTORY'
    elif path.name == 'FAILED_CASES.csv': kind = 'FAILED_CASES'
    elif path.name in {'LOCKED_CASE_MANIFEST.csv', 'LOCKED_FOLD_MANIFEST.csv', 'FULL_RUN_STATUS.json'}: kind = 'IDENTITY'
    if kind:
        selected.append({'artifact_kind': kind, 'relative_path': relative, 'size_bytes': path.stat().st_size, 'sha256': digest(path)})

counts = {}
for row in selected: counts[row['artifact_kind']] = counts.get(row['artifact_kind'], 0) + 1
assert counts['HELD_OUT_P0'] == 40, counts
assert counts['FOLD_CHECKPOINT'] == 5, counts
assert counts['MAIN_METRICS'] == 1 and counts['MAIN_TRAJECTORY'] == 1
selected.sort(key=lambda row: (row['artifact_kind'], row['relative_path']))
out = Path('/kaggle/working')
with (out / 'FROZEN_ARTIFACT_HASHES.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['artifact_kind','relative_path','size_bytes','sha256'])
    writer.writeheader(); writer.writerows(selected)
identity = {'source_kernel':'jeechangxin/pcc-leakage-free-rerun-2026','source_version':8,'artifact_root':str(artifact_root),'counts':counts,'status':status}
(out / 'FROZEN_RUN_IDENTITY.json').write_text(json.dumps(identity, indent=2) + '\n')
print(json.dumps(identity, indent=2))
